In [1]:
import sys
import os
import json
import torch

import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix, save_npz, load_npz
from scipy.stats import mannwhitneyu

In [2]:
interpro_annotations_nonzero = pd.read_csv("metadata/interpro_entry_list_mapping_nonzero.csv")
ptn_fam_nonzero_tensor = torch.load("metadata/ptn_fam_tensor_nonzero.pt")

latents_family_effect_size = torch.load("/mnt/polished-lake/home/connor/plm_circuits/notebooks/domain_corr/data/latents_property_top_q_effect_size.pt")
latents_family_pvals = torch.load("/mnt/polished-lake/home/connor/plm_circuits/notebooks/domain_corr/data/latents_property_top_q_pvals.pt")

In [3]:
print(f"Interpro Annotations: {interpro_annotations_nonzero.shape}")
print(f"ptn_fam_nonzero_tensor: {ptn_fam_nonzero_tensor.shape}")
print(f"Effect sizes: {latents_family_effect_size.shape}")
print(f"MWU pvals: {latents_family_pvals.shape}")

Interpro Annotations: (10096, 3)
ptn_fam_nonzero_tensor: torch.Size([10000, 10096])
Effect sizes: torch.Size([383, 10096, 4])
MWU pvals: torch.Size([383, 10096])


In [4]:
interpro_annotations_nonzero.head()

,ENTRY_AC,ENTRY_TYPE,ENTRY_NAME
0,IPR000126,Active_site,"Serine proteases, V8 family, serine active site"
1,IPR000169,Active_site,"Cysteine peptidase, cysteine active site"
2,IPR000189,Active_site,"Prokaryotic transglycosylase, active site"
3,IPR001252,Active_site,"Malate dehydrogenase, active site"
4,IPR001345,Active_site,"Phosphoglycerate/bisphosphoglycerate mutase, a..."


In [5]:
interpro_annotations_nonzero[interpro_annotations_nonzero['ENTRY_NAME'].str.contains("SAM")]

,ENTRY_AC,ENTRY_TYPE,ENTRY_NAME
324,IPR018063,Conserved_site,"SAM-dependent methyltransferase RsmI, conserve..."
949,IPR001678,Domain,SAM-dependent methyltransferase RsmB-F/NOP2-ty...
1287,IPR004107,Domain,"Integrase, SAM-like, N-terminal"
1545,IPR006638,Domain,"Elp3/MiaA/NifB-like, radical SAM core domain"
1601,IPR007197,Domain,Radical SAM
2467,IPR022642,Domain,"MCP methyltransferase, CheR-type, SAM-binding ..."
2714,IPR030382,Domain,SAM-dependent methyltransferase TRM5/TYW2-type
3034,IPR037635,Domain,"VTS1, SAM domain"
3247,IPR042650,Domain,"ASZ1, SAM domain"
3401,IPR047238,Domain,Ankyrin repeat and SAM domain-containing prote...


In [52]:
interpro_ids_for_manual = {"top2": ["IPR001757", # HATPase
                                    "IPR006086", # XPG-I domain
                                    "IPR001752"  # Kinesin
                                   ],
                           "metx":  [
                               "IPR029058", # Alpha/Beta hydrolase fold
                               "",
                               "",
                           ]}

In [ ]:
#with open("/mnt/polished-lake/home/connor/plm_circuits/results/layer_latent_dicts/layer_latent_dict_MetXA_0.70.json", 'r') as file:
#    metx_latents = json.load(file)

#with open("/mnt/polished-lake/home/connor/plm_circuits/results/layer_latent_dicts/layer_latent_dict_Top2_0.70.json", 'r') as file:
#    top2_latents = json.load(file)

with open("/mnt/polished-lake/home/connor/plm_circuits/results/layer_latent_dicts/layer_latent_dict_2PKEA_0.70.json", 'r') as file:
   pkea_latents  = json.load(file)

In [7]:
#offsets = {k: i*4096 for i,k in enumerate(metx_latents.keys())}
offsets = {k: i*4096 for i,k in enumerate(pkea_latents.keys())}
#full_indices_metx = [j+offsets[layer_id] for layer_id in metx_latents.keys() for j in metx_latents[layer_id]]
#full_indices_top2 = [j+offsets[layer_id] for layer_id in top2_latents.keys() for j in top2_latents[layer_id]]
full_indices_pkea = [j+offsets[layer_id] for layer_id in pkea_latents.keys() for j in pkea_latents[layer_id]]
#latent_ids_both_proteins = list(set(full_indices_top2+full_indices_metx+full_indices_pkea))
latent_ids_both_proteins = full_indices_pkea
latent_ids_both_proteins.sort()
len(latent_ids_both_proteins) # yes, only 383 unique

262

In [8]:
latent_ids_both_proteins[0:10]

[104, 794, 897, 1080, 1474, 1509, 1525, 2119, 2154, 2487]

In [9]:
n_tests = 28672*10096

In [13]:
n_tests

289472512

In [10]:
np.argmin(latents_family_pvals[0])

tensor(8760)

In [11]:
annotated_latents = []
for latent_i in range(latents_family_pvals.shape[0]):
    for family_j in range(latents_family_pvals.shape[1]):
        if latents_family_pvals[latent_i,family_j] < (0.05/n_tests):
            if latents_family_effect_size[latent_i,family_j,1] < 0.05:
                l2fc = np.log2(latents_family_effect_size[latent_i,family_j,0]/ (latents_family_effect_size[latent_i,family_j,1]+0.00001))
                if l2fc >= 4:  # latents_family_effect_size[latent_i,family_j,0] > 1.05: #
                    annotated_latents += [latent_ids_both_proteins[latent_i]]
                    print(f"Latent {latent_ids_both_proteins[latent_i]} is sig assoc w family {family_j}")
                    entry_name = interpro_annotations_nonzero.iloc[family_j]["ENTRY_NAME"]
                    print(f"    Family {entry_name}")
                    print(f"    Mean in-family: {latents_family_effect_size[latent_i,family_j,0]}")
                    print(f"    Mean out-of-family: {latents_family_effect_size[latent_i,family_j,1]}")
                    print(f"    Log2FC: {np.log2(latents_family_effect_size[latent_i,family_j,0]/latents_family_effect_size[latent_i,family_j,1])}")

Latent 11943 is sig assoc w family 57
    Family Carboxylesterase type B, active site
    Mean in-family: 1.8721842765808105
    Mean out-of-family: 0.03769707679748535
    Log2FC: 5.634125709533691
Latent 11943 is sig assoc w family 64
    Family Glyceraldehyde 3-phosphate dehydrogenase, active site
    Mean in-family: 0.6408420205116272
    Mean out-of-family: 0.037885647267103195
    Log2FC: 4.080245494842529
Latent 11943 is sig assoc w family 158
    Family Porphobilinogen deaminase, dipyrromethane cofactor binding site
    Mean in-family: 0.7127278447151184
    Mean out-of-family: 0.0377749539911747
    Log2FC: 4.237849235534668
Latent 11943 is sig assoc w family 183
    Family AB hydrolase 4, conserved site
    Mean in-family: 4.200439453125
    Mean out-of-family: 0.03741481155157089
    Log2FC: 6.810786724090576
Latent 11943 is sig assoc w family 239
    Family Recombinase, conserved site
    Mean in-family: 2.3385417461395264
    Mean out-of-family: 0.037557121366262436
    Lo

In [12]:
def to_flattened_id(layer_id: int, lat_id: int) -> int:
    if layer_id % 4 != 0:
        raise ValueError("layer_id must be a multiple of 4 (e.g., 4, 8, 12, ...)")
    if not (0 <= lat_id < 4096):
        raise ValueError("lat_id must be in [0, 4096)")
    return ((layer_id // 4) - 1) * 4096 + lat_id


for layer, latents in pkea_latents.items(): # ONLY PKEA FOR NOW
    for lat_id in latents:
        print("="*100)
        print(layer, lat_id)
        flattened_id = to_flattened_id(int(layer), int(lat_id))
        latent_i = latent_ids_both_proteins.index(flattened_id)

        for family_j in range(latents_family_pvals.shape[1]):
            if latents_family_pvals[latent_i,family_j] < (0.05/n_tests):
                if latents_family_effect_size[latent_i,family_j,1] < latents_family_effect_size[latent_i,family_j,0]:
                    l2fc = np.log2(latents_family_effect_size[latent_i,family_j,0]/ (latents_family_effect_size[latent_i,family_j,1]+0.00001))

                    print(f"    Family {interpro_annotations_nonzero.iloc[family_j]['ENTRY_NAME']}")
                    print(f"    Mean in-family: {latents_family_effect_size[latent_i,family_j,0]}")
                    print(f"    Mean out-of-family: {latents_family_effect_size[latent_i,family_j,1]}")
                    print(f"    Log2FC: {np.log2(latents_family_effect_size[latent_i,family_j,0]/latents_family_effect_size[latent_i,family_j,1])}")

4 3501
    Family Serine/threonine-protein kinase, active site
    Mean in-family: 6.779625415802002
    Mean out-of-family: 6.260067939758301
    Log2FC: 0.1150272935628891
    Family Protein kinase, ATP binding site
    Mean in-family: 6.797611236572266
    Mean out-of-family: 6.260136604309082
    Log2FC: 0.11883369088172913
    Family Protein kinase domain
    Mean in-family: 6.79779052734375
    Mean out-of-family: 6.259378433227539
    Log2FC: 0.11904654651880264
    Family Helicase superfamily 1/2, ATP-binding domain
    Mean in-family: 6.698356628417969
    Mean out-of-family: 6.259875297546387
    Log2FC: 0.09767322987318039
    Family Protein kinase-like domain superfamily
    Mean in-family: 6.761026382446289
    Mean out-of-family: 6.258852481842041
    Log2FC: 0.11134409159421921
4 897
    Family Serine/threonine-protein kinase, active site
    Mean in-family: 4.281973838806152
    Mean out-of-family: 3.8640780448913574
    Log2FC: 0.14815178513526917
    Family Protein ki

In [14]:
import numpy as np
import pandas as pd
import torch

EPS = 1e-9  # tiny epsilon for ratios

def as_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def build_top3_table(pkea_latents, k=3, alpha=0.05):
    rows = []
    bonf = alpha / n_tests

    # Ensure these big tensors/arrays are NumPy
    P = as_numpy(latents_family_pvals)                   # [n_latents, n_families]
    E = as_numpy(latents_family_effect_size)             # [:, :, 0]=mean_in, [:, :, 1]=mean_out

    for layer, latents in pkea_latents.items():
        for lat_id in latents:
            flattened_id = to_flattened_id(int(layer), int(lat_id))
            latent_i = latent_ids_both_proteins.index(flattened_id)

            pvals    = P[latent_i, :]
            mean_in  = E[latent_i, :, 0]
            mean_out = E[latent_i, :, 1]

            # numeric hygiene
            ratio = (mean_in + EPS) / (mean_out + EPS)
            mask = np.isfinite(ratio) & (pvals < bonf) & (mean_in > mean_out)
            idx = np.where(mask)[0]
            if idx.size == 0:
                continue

            # DESC sort without [::-1] to avoid the "step must be > 0" issue
            order = np.argsort(-ratio[idx])  # minus sign = descending
            top = idx[order[:k]]

            for j in top:
                rows.append({
                    "Layer": layer,
                    "Latent": lat_id,
                    "Family_ID": int(j),
                    "Family": interpro_annotations_nonzero.iloc[int(j)]["ENTRY_NAME"], #[:30],
                    "pval": float(pvals[j]),
                    "mean_in": float(mean_in[j]),
                    "mean_out": float(mean_out[j]),
                    "ratio_in_out": float(ratio[j]),
                })

    if not rows:
        return pd.DataFrame(columns=["Layer","Latent","Family_ID","Family","pval",
                                     "mean_in","mean_out","ratio_in_out"])

    df = pd.DataFrame(rows)#.sort_values(
        # ["ratio_in_out", "pval"], ascending=[False, True], ignore_index=True
    # )
    # Nice display tweaks
    df.insert(0, "LatentID", df["Layer"].astype(str)+":"+df["Latent"].astype(str))
    df["pval_fmt"] = df["pval"].map(lambda x: f"{x:.2e}")
    df["mean_in"] = df["mean_in"].map(lambda x: round(x, 3))
    df["mean_out"] = df["mean_out"].map(lambda x: round(x, 3))
    df["ratio_in_out"] = df["ratio_in_out"].map(lambda x: round(x, 3))

    cols = ["LatentID","Family","pval_fmt","mean_in","mean_out","ratio_in_out","Family_ID"]
    return df[cols]
    return df

# run
df_top3 = build_top3_table(pkea_latents, k=3, alpha=0.05)
print(df_top3.to_string(index=False))

LatentID                                                                                    Family pval_fmt  mean_in  mean_out  ratio_in_out  Family_ID
  4:3501                                                                     Protein kinase domain 1.36e-18    6.798     6.259         1.086        767
  4:3501                                                          Protein kinase, ATP binding site 7.27e-15    6.798     6.260         1.086        126
  4:3501                                              Serine/threonine-protein kinase, active site 1.59e-14    6.780     6.260         1.083         21
   4:897                                                                  Homeobox, conserved site 7.26e-12    4.519     3.864         1.169        306
   4:897                                                                               Homeodomain 1.20e-13    4.483     3.864         1.160        897
   4:897                                                               Homedomain-like s

In [ ]:
# print 

In [11]:
# Switching from Bonferroni to Benjamini-Hochberg FDR correction
# This is more appropriate for exploratory analyses with millions of tests

from statsmodels.stats.multitest import multipletests

print("=" * 80)
print("DOMAIN DETECTOR ANALYSIS WITH BH-FDR CORRECTION")
print("=" * 80)

# Step 1: Collect all p-values and their corresponding indices
print("\n1. Collecting p-values from all latent-family pairs...")
pval_data = []
for latent_i in range(latents_family_pvals.shape[0]):
    for family_j in range(latents_family_pvals.shape[1]):
        pval_data.append({
            'pval': float(latents_family_pvals[latent_i, family_j]),
            'latent_i': latent_i,
            'family_j': family_j
        })
print(f"   Total tests: {len(pval_data):,}")

# Step 2: Apply Benjamini-Hochberg FDR correction
print("\n2. Applying BH-FDR correction (alpha = 0.05)...")
pvals = [d['pval'] for d in pval_data]
rejected, pvals_adjusted, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

# Add adjusted p-values back to our data structure
for i, d in enumerate(pval_data):
    d['pval_adjusted'] = pvals_adjusted[i]
    d['significant'] = rejected[i]

print(f"   Tests passing FDR < 0.05: {sum(rejected):,}")

# Step 3: Apply effect size filters to significant associations
print("\n3. Applying effect size filters:")
print("   - Out-of-family mean < 0.05 (specificity filter)")
print("   - Log2 fold change >= 4 (effect size filter)")

annotated_latents_fdr = []
significant_associations = []

for d in pval_data:
    if d['significant']:  # Passes FDR correction
        latent_i, family_j = d['latent_i'], d['family_j']

        # Check effect size criteria
        mean_in_family = float(latents_family_effect_size[latent_i, family_j, 0])
        mean_out_family = float(latents_family_effect_size[latent_i, family_j, 1])

        if mean_out_family < 0.05:  # Specificity filter
            l2fc = np.log2(mean_in_family / (mean_out_family + 0.00001))

            if l2fc >= 4:  # Effect size filter
                latent_id = latent_ids_both_proteins[latent_i]
                annotated_latents_fdr.append(latent_id)

                # Store for summary
                significant_associations.append({
                    'latent_id': latent_id,
                    'family_j': family_j,
                    'mean_in': mean_in_family,
                    'mean_out': mean_out_family,
                    'l2fc': l2fc,
                    'pval_adj': d['pval_adjusted']
                })

# Step 4: Summary statistics
print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)

# Compare with Bonferroni
bonf_threshold = 0.05 / n_tests
print(f"\nComparison of correction methods:")
print(f"  Bonferroni threshold: {bonf_threshold:.2e}")
print(f"  Minimum adjusted p-value (BH-FDR): {min(pvals_adjusted):.2e}")
print(f"  Maximum significant adjusted p-value (BH-FDR): {max([d['pval_adjusted'] for d in pval_data if d['significant']]):.2e}")

print(f"\nDomain detectors found:")
print(f"  Total associations: {len(annotated_latents_fdr):,}")
print(f"  Unique latents: {len(set(annotated_latents_fdr)):,}")

# Show breakdown by protein
metx_latents_fdr = [l for l in set(annotated_latents_fdr) if l in full_indices_metx]
top2_latents_fdr = [l for l in set(annotated_latents_fdr) if l in full_indices_top2]

print(f"\nBreakdown by protein:")
print(f"  MetX latents: {len(metx_latents_fdr)} / {len(set(full_indices_metx))} ({100*len(metx_latents_fdr)/len(set(full_indices_metx)):.1f}%)")
print(f"  Top2 latents: {len(top2_latents_fdr)} / {len(set(full_indices_top2))} ({100*len(top2_latents_fdr)/len(set(full_indices_top2)):.1f}%)")

# Show top 10 strongest associations
print("\n" + "=" * 80)
print("TOP 10 STRONGEST ASSOCIATIONS (by Log2FC)")
print("=" * 80)

significant_associations.sort(key=lambda x: x['l2fc'], reverse=True)
for i, assoc in enumerate(significant_associations[:10], 1):
    family_info = interpro_annotations_nonzero.iloc[assoc['family_j']]
    layer = ((assoc['latent_id'] // 4096) + 1) * 4
    latent_within_layer = assoc['latent_id'] % 4096

    print(f"\n{i}. Layer {layer}, Latent {latent_within_layer} (Global ID: {assoc['latent_id']})")
    print(f"   Domain: {family_info['ENTRY_NAME']}")
    print(f"   InterPro ID: {family_info['ENTRY_AC']}")
    print(f"   Mean activation in-family: {assoc['mean_in']:.3f}")
    print(f"   Mean activation out-of-family: {assoc['mean_out']:.4f}")
    print(f"   Log2 Fold Change: {assoc['l2fc']:.2f}")
    print(f"   Adjusted p-value: {assoc['pval_adj']:.2e}")

DOMAIN DETECTOR ANALYSIS WITH BH-FDR CORRECTION

1. Collecting p-values from all latent-family pairs...
   Total tests: 3,866,768

2. Applying BH-FDR correction (alpha = 0.05)...
   Tests passing FDR < 0.05: 342,499

3. Applying effect size filters:
   - Out-of-family mean < 0.05 (specificity filter)
   - Log2 fold change >= 4 (effect size filter)


/tmp/ipykernel_943005/1609149312.py:51: RuntimeWarning: divide by zero encountered in log2
  l2fc = np.log2(mean_in_family / (mean_out_family + 0.00001))



RESULTS SUMMARY

Comparison of correction methods:
  Bonferroni threshold: 1.73e-10
  Minimum adjusted p-value (BH-FDR): 0.00e+00
  Maximum significant adjusted p-value (BH-FDR): 5.00e-02

Domain detectors found:
  Total associations: 188
  Unique latents: 1

Breakdown by protein:
  MetX latents: 1 / 273 (0.4%)
  Top2 latents: 0 / 134 (0.0%)

TOP 10 STRONGEST ASSOCIATIONS (by Log2FC)

1. Layer 8, Latent 488 (Global ID: 4584)
   Domain: Uncharacterised protein family UPF0227/Esterase YqiA
   InterPro ID: IPR008886
   Mean activation in-family: 6.711
   Mean activation out-of-family: 0.0369
   Log2 Fold Change: 7.51
   Adjusted p-value: 6.08e-12

2. Layer 8, Latent 488 (Global ID: 4584)
   Domain: Uncharacterised protein family UPF0227
   InterPro ID: IPR022987
   Mean activation in-family: 6.711
   Mean activation out-of-family: 0.0369
   Log2 Fold Change: 7.51
   Adjusted p-value: 6.08e-12

3. Layer 8, Latent 488 (Global ID: 4584)
   Domain: Serine aminopeptidase, S33
   InterPro ID: 

In [15]:
ann_latents_list = list(set(annotated_latents))

In [16]:
len(ann_latents_list)

1

In [17]:
len(set(full_indices_metx))

273

In [18]:
len(set(full_indices_metx).intersection(ann_latents_list))

1

In [19]:
len(set(full_indices_top2))

134

In [20]:
len(set(full_indices_top2).intersection(ann_latents_list))

0

In [34]:
# For a table:
# Layer | Latent ID (#) | IntroPro ID | InterPro Name | out-of-family | in-family | Bonf p-value

In [35]:
metx_results_df_dict = {}
for latent_i in range(latents_family_pvals.shape[0]):
    for family_j in range(latents_family_pvals.shape[1]):
        if latents_family_pvals[latent_i,family_j] < (0.05/n_tests):
            if latents_family_effect_size[latent_i,family_j,1] < 0.05:
                l2fc = np.log2(latents_family_effect_size[latent_i,family_j,0]/(latents_family_effect_size[latent_i,family_j,1]+0.00001))
                if l2fc >= 4:  # latents_family_effect_size[latent_i,family_j,0] > 1.05:

                    df_id = str(latent_i)+"_"+str(family_j)
                    cur_lat_id = latent_ids_both_proteins[latent_i]
                    layer_id = ((cur_lat_id // 4096)+1)*4
                    lat_id = cur_lat_id % 4096

                    case_ptn = "MetX"
                    if latent_ids_both_proteins[latent_i] in full_indices_top2:
                        case_ptn="Top2"

                    metx_results_df_dict[df_id] = {
                        "Case": case_ptn,
                        "Layer": layer_id,
                        "Latent ID": lat_id,
                        "InterPro ID": interpro_annotations_nonzero.iloc[family_j]["ENTRY_AC"],
                        "InterPro Name": interpro_annotations_nonzero.iloc[family_j]["ENTRY_NAME"],
                        "A_in": float(latents_family_effect_size[latent_i,family_j,0]),
                        "A_out": float(latents_family_effect_size[latent_i,family_j,1]),
                        "L2FC": float(l2fc),
                        "Bonferroni adj. p-value": float(latents_family_pvals[latent_i,family_j]*n_tests)
                    }

/tmp/ipykernel_331029/507448697.py:6: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  l2fc = np.log2(latents_family_effect_size[latent_i,family_j,0]/(latents_family_effect_size[latent_i,family_j,1]+0.00001))
/tmp/ipykernel_331029/507448697.py:6: RuntimeWarning: divide by zero encountered in log2
  l2fc = np.log2(latents_family_effect_size[latent_i,family_j,0]/(latents_family_effect_size[latent_i,family_j,1]+0.00001))


In [36]:
df_metx_results = pd.DataFrame.from_dict(metx_results_df_dict, orient='index')
df_metx_results.head()

,Case,Layer,Latent ID,InterPro ID,InterPro Name,A_in,A_out,L2FC,Bonferroni adj. p-value
13_3824,Top2,4,1297,IPR000473,Large ribosomal subunit protein bL36,0.073116,0.003320,4.456454,6.988324e-07
13_5069,Top2,4,1297,IPR005996,"Large ribosomal subunit protein uL30, bacteria...",0.057346,0.003392,4.075256,7.427055e-05
13_9135,Top2,4,1297,IPR035977,Large ribosomal subunit protein bL36 superfamily,0.073116,0.003320,4.456454,6.988324e-07
46_128,Top2,4,3717,IPR018064,"Metallothionein, vertebrate, metal binding site",0.040059,0.001214,5.032932,5.300378e-05
46_540,Top2,4,3717,IPR020939,"Large ribosomal subunit protein bL34, conserve...",0.019646,0.001192,4.030718,6.362296e-09


In [38]:
df_metx_results.query("Case == 'MetX' and L2FC > 5 and A_in >= 0.5")#.sort_values(by="Bonferroni adj. p-value")

,Case,Layer,Latent ID,InterPro ID,InterPro Name,A_in,A_out,L2FC,Bonferroni adj. p-value
80_659,MetX,12,2112,IPR000073,Alpha/beta hydrolase fold-1,0.604529,0.007763,6.281209,3.070172e-14
80_9059,MetX,12,2112,IPR029058,Alpha/Beta hydrolase fold,0.559808,0.005224,6.740910,0.000000e+00
82_5515,MetX,12,2302,IPR010084,Beta-hydroxyacyl-(acyl-carrier-protein) dehydr...,0.549541,0.002915,7.553611,2.303711e-04
82_5813,MetX,12,2302,IPR013114,"Beta-hydroxydecanoyl thiol ester dehydrase, Fa...",0.565334,0.002730,7.688618,8.962103e-08
136_659,MetX,16,1504,IPR000073,Alpha/beta hydrolase fold-1,1.283272,0.026213,5.612868,8.708278e-06
136_9059,MetX,16,1504,IPR029058,Alpha/Beta hydrolase fold,1.323345,0.020013,6.046389,0.000000e+00
145_9059,MetX,16,2175,IPR029058,Alpha/Beta hydrolase fold,1.561086,0.022779,6.098062,0.000000e+00
170_4594,MetX,16,3765,IPR003824,Undecaprenyl-diphosphatase UppP,1.132152,0.032733,5.111751,2.038676e-06
177_797,MetX,20,142,IPR000873,AMP-dependent synthetase/ligase domain,2.935064,0.044304,6.049480,1.535564e-02
177_2169,MetX,20,142,IPR015590,Aldehyde dehydrogenase domain,2.456921,0.043235,5.828182,1.386195e-07


In [31]:
df_metx_results.query("Case == 'Top2'")#.sort_values(by="Bonferroni adj. p-value")

,Case,Layer,Latent ID,InterPro ID,InterPro Name,Activation w/ domain,Activation w/o domain,L2FC,Bonferroni adj. p-value
46_9135,Top2,4,3717,IPR035977,Large ribosomal subunit protein bL36 superfamily,0.082034,0.001080,6.234466,0.000000
46_3824,Top2,4,3717,IPR000473,Large ribosomal subunit protein bL36,0.082034,0.001080,6.234466,0.000000
72_781,Top2,12,1204,IPR000793,"ATP synthase, alpha subunit, C-terminal",0.297229,0.004520,6.036056,0.000000
72_672,Top2,12,1204,IPR000194,"ATPase, F1/V1/A1 complex, alpha/beta subunit, ...",0.276573,0.004105,6.070641,0.000000
72_58,Top2,12,1204,IPR020003,"ATPase, alpha/beta subunit, nucleotide-binding...",0.276573,0.004105,6.070641,0.000000
...,...,...,...,...,...,...,...,...,...
214_9912,Top2,20,1966,IPR042221,"Leucyl/phenylalanyl-tRNA-protein transferase, ...",6.532168,0.023353,8.127212,0.021919
214_1058,Top2,20,1966,IPR002500,Phosphoadenosine phosphosulphate reductase domain,1.451269,0.026912,5.752396,0.028800
84_669,Top2,12,2474,IPR000182,GNAT domain,1.381740,0.022060,5.968279,0.034206
214_1909,Top2,20,1966,IPR012795,"tRNA(Ile)-lysidine synthase, N-terminal",0.617451,0.027496,4.488513,0.042795


In [39]:
df_metx_results.to_csv('circuit_domain_detectors.csv', index=False)

In [40]:
print(df_metx_results.query("Case == 'Top2'").drop("Case", axis=1).to_latex(
    index=False, float_format="{:.2g}".format))

\begin{tabular}{rrllrrr}
\toprule
Layer & Latent ID & InterPro ID & InterPro Name & Activation w/ domain & Activation w/o domain & Bonferroni adj. p-value \\
\midrule
12 & 1204 & IPR014762 & DNA mismatch repair, conserved site & 1.5 & 0.0045 & 1.6e-07 \\
12 & 1204 & IPR019805 & Heat shock protein Hsp90, conserved site & 1.5 & 0.0042 & 2.9e-13 \\
12 & 1204 & IPR013507 & DNA mismatch repair protein, S5 domain 2-like & 1.5 & 0.0045 & 1.6e-07 \\
12 & 1204 & IPR014790 & MutL, C-terminal, dimerisation & 1.5 & 0.0045 & 1.6e-07 \\
12 & 1204 & IPR020575 & Heat shock protein Hsp90, N-terminal & 1.4 & 0.0041 & 3.9e-16 \\
12 & 1204 & IPR001404 & Heat shock protein Hsp90 family & 1.4 & 0.0041 & 3.9e-16 \\
12 & 1204 & IPR002099 & DNA mismatch repair protein MutL/Mlh/PMS & 1.5 & 0.0045 & 1.6e-07 \\
12 & 1204 & IPR020667 & DNA mismatch repair protein, MutL & 1.5 & 0.0045 & 1.6e-07 \\
12 & 1204 & IPR038973 & DNA mismatch repair protein MutL/Mlh/Pms-like & 1.5 & 0.0045 & 1.6e-07 \\
12 & 1204 & IPR037196

In [41]:
print(df_metx_results.query("Case == 'MetX'").drop("Case", axis=1).to_latex(
    index=False, float_format="{:.2g}".format))

\begin{tabular}{rrllrrr}
\toprule
Layer & Latent ID & InterPro ID & InterPro Name & Activation w/ domain & Activation w/o domain & Bonferroni adj. p-value \\
\midrule
16 & 1504 & IPR000073 & Alpha/beta hydrolase fold-1 & 1.3 & 0.026 & 8.7e-06 \\
16 & 1504 & IPR029058 & Alpha/Beta hydrolase fold & 1.3 & 0.02 & 0 \\
16 & 2175 & IPR029058 & Alpha/Beta hydrolase fold & 1.6 & 0.023 & 0 \\
16 & 2836 & IPR029058 & Alpha/Beta hydrolase fold & 1.3 & 0.045 & 3e-35 \\
16 & 3765 & IPR003824 & Undecaprenyl-diphosphatase UppP & 1.1 & 0.033 & 2e-06 \\
20 & 142 & IPR000873 & AMP-dependent synthetase/ligase domain & 2.9 & 0.044 & 0.015 \\
20 & 142 & IPR015590 & Aldehyde dehydrogenase domain & 2.5 & 0.043 & 1.4e-07 \\
20 & 142 & IPR016161 & Aldehyde/histidinol dehydrogenase & 2.5 & 0.042 & 1.7e-10 \\
20 & 142 & IPR016162 & Aldehyde dehydrogenase, N-terminal & 2.5 & 0.043 & 1.4e-07 \\
20 & 142 & IPR016163 & Aldehyde dehydrogenase, C-terminal & 2.5 & 0.043 & 1.4e-07 \\
20 & 142 & IPR045851 & AMP-binding e